# Promote a Model Version — Dev Catalog to Prod Catalog
Copies **one** model version from the dev UC model into a destination UC model in the prod catalog.
Only the selected version is copied — not the full version history.
All settings are hardcoded in the **Promotion Settings** cell (`SOURCE_MODEL`, `TARGET_MODEL`, version/alias); no widgets.
Run by a service principal with access to both catalogs.

**Source selection:** set `SOURCE_VERSION` to copy a specific version; leave it empty to copy the version behind `SOURCE_ALIAS` (default `@champion`).

**Copy:** `MlflowClient.copy_model_version` registers the selected version as the next version number of the destination model (created on first copy).

The destination version gets lineage tags (`source_model`, `source_version`, `source_run_id`) and, optionally, an alias (default `champion`).
Re-running is idempotent: if that exact source version was already promoted, the existing destination version is reused.

In [0]:
import time
import mlflow
from mlflow.tracking import MlflowClient
from src.config import CFG
from src.utils import get_logger

logger = get_logger("models.promote_model_to_prod")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(CFG.experiment_name)  # one shared experiment for the whole repo
client = MlflowClient()  # inherits the registry URI set above

In [0]:
# ── Promotion settings (hardcoded; run by an SPN with access to both catalogs) ──
# Source = dev catalog, destination = prod catalog (bundle prod target: mlops_prod.siva_borusu).
SOURCE_MODEL = "mlops.siva_borusu.advisor_risk_model_pyfunc"
TARGET_MODEL = "mlops.siva_borusu.advisor_risk_model_pyfunc_prod_replica"

# Which source version to copy: SOURCE_VERSION wins when set, otherwise SOURCE_ALIAS is used.
SOURCE_VERSION = ""            # e.g. "3"; "" = use SOURCE_ALIAS
SOURCE_ALIAS = "champion"

TARGET_ALIAS = "champion"      # alias set on the destination version; "" = none

if SOURCE_VERSION and not str(SOURCE_VERSION).isdigit():
    raise ValueError(f"SOURCE_VERSION must be a number, got '{SOURCE_VERSION}'")
if not SOURCE_VERSION and not SOURCE_ALIAS:
    raise ValueError("Set SOURCE_VERSION or SOURCE_ALIAS.")
if len(SOURCE_MODEL.split(".")) != 3:
    raise ValueError(f"SOURCE_MODEL must be catalog.schema.model, got '{SOURCE_MODEL}'")
if len(TARGET_MODEL.split(".")) != 3:
    raise ValueError(f"TARGET_MODEL must be catalog.schema.model, got '{TARGET_MODEL}'")
if TARGET_MODEL == SOURCE_MODEL:
    raise ValueError("Destination model must differ from the source model.")

logger.info(
    f"Source: {SOURCE_MODEL} ({'v' + str(SOURCE_VERSION) if SOURCE_VERSION else '@' + SOURCE_ALIAS}) "
    f"-> Destination: {TARGET_MODEL}"
)

In [0]:
if SOURCE_VERSION:
    src_mv = client.get_model_version(SOURCE_MODEL, str(SOURCE_VERSION))
else:
    src_mv = client.get_model_version_by_alias(SOURCE_MODEL, SOURCE_ALIAS)

SRC_URI = f"models:/{SOURCE_MODEL}/{src_mv.version}"
logger.info(f"Selected source: {SRC_URI} (run_id={src_mv.run_id})")

In [0]:
def find_existing_promotion():
    """Return a destination version already promoted from this exact source version, if any."""
    try:
        versions = client.search_model_versions(f"name='{TARGET_MODEL}'")
    except Exception:
        return None  # destination model does not exist yet
    for mv in versions:
        tags = client.get_model_version(TARGET_MODEL, mv.version).tags
        if tags.get("source_model") == SOURCE_MODEL and tags.get("source_version") == str(src_mv.version):
            return client.get_model_version(TARGET_MODEL, mv.version)
    return None


def wait_until_ready(version: str, timeout_s: int = 600):
    """Poll the destination version until registration finishes."""
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        mv = client.get_model_version(TARGET_MODEL, version)
        if mv.status == "READY":
            return mv
        if mv.status == "FAILED_REGISTRATION":
            raise RuntimeError(f"Registration failed for {TARGET_MODEL} v{version}: {mv.status_message}")
        time.sleep(10)
    raise TimeoutError(f"{TARGET_MODEL} v{version} not READY after {timeout_s}s")


existing = find_existing_promotion()
if existing:
    logger.info(f"{SRC_URI} already promoted as {TARGET_MODEL} v{existing.version} — reusing it.")
    dst_mv = existing
else:
    new_mv = client.copy_model_version(src_model_uri=SRC_URI, dst_name=TARGET_MODEL)
    dst_mv = wait_until_ready(new_mv.version)
    logger.info(f"Copied {SRC_URI} -> {TARGET_MODEL} v{dst_mv.version}")

In [0]:
lineage_tags = {
    "source_model": SOURCE_MODEL,
    "source_version": str(src_mv.version),
    "source_run_id": src_mv.run_id or "",
}
for k, v in lineage_tags.items():
    client.set_model_version_tag(TARGET_MODEL, dst_mv.version, k, v)

client.update_model_version(
    TARGET_MODEL, dst_mv.version,
    description=f"Promoted from {SRC_URI}.",
)

if TARGET_ALIAS:
    client.set_registered_model_alias(TARGET_MODEL, TARGET_ALIAS, dst_mv.version)
    logger.info(f"Set @{TARGET_ALIAS} -> {TARGET_MODEL} v{dst_mv.version}")

In [0]:
dst_uri = f"models:/{TARGET_MODEL}/{dst_mv.version}"
src_info = mlflow.models.get_model_info(SRC_URI)
dst_info = mlflow.models.get_model_info(dst_uri)

if str(src_info.signature) != str(dst_info.signature):
    raise AssertionError(f"Signature mismatch:\nsource={src_info.signature}\ndest={dst_info.signature}")
if set(src_info.flavors) != set(dst_info.flavors):
    raise AssertionError(f"Flavor mismatch: source={set(src_info.flavors)} dest={set(dst_info.flavors)}")

dst_final = client.get_model_version(TARGET_MODEL, dst_mv.version)
if TARGET_ALIAS and TARGET_ALIAS not in dst_final.aliases:
    raise AssertionError(f"Alias @{TARGET_ALIAS} not set on {dst_uri}")

logger.info(
    f"Verified {dst_uri}: flavors={sorted(dst_info.flavors)}, aliases={dst_final.aliases}, "
    f"tags={ {k: dst_final.tags.get(k) for k in lineage_tags} }"
)
dbutils.jobs.taskValues.set("promoted_version", str(dst_mv.version))